## Cardinality

### What is cardinality? (Simple explanation)

**Cardinality** refers to the **number of unique values** in a categorical variable.

**Examples:**

- `Gender` → 2 unique values → **Low cardinality**
- `Country` → ~200 unique values → **Medium cardinality**
- `User_ID` → millions of unique values → **High cardinality**

---

### Why is high cardinality a problem?

High-cardinality features can cause several issues in feature engineering and modeling.

#### 1. Explosion of features

When using encoding methods like **one-hot encoding**, each unique category becomes a new column.

- 10 categories → 10 columns (manageable)
- 10,000 categories → 10,000 columns (problematic)

This leads to:

- High memory usage
- Slower training
- Sparse data

#### 2. Overfitting

High-cardinality features often allow the model to **memorize** the training data instead of learning general patterns.

Example:

- `User_ID` uniquely identifies users
- Model learns user-specific behavior that does not generalize

#### 3. Poor generalization

Many categories appear **only a few times**.

- The model has little data to learn from each category
- Performance drops on unseen data

#### 4. Increased model complexity

More unique values increase:

- Model size
- Training time
- Difficulty of interpretation

#### 5. Encoding challenges

Common encoders struggle with high cardinality:

- One-hot encoding becomes infeasible
- Label encoding may introduce false ordering
- Target encoding risks leakage if not handled carefully

### Industry perspective

In practice:

- High-cardinality features are **carefully engineered**
- Often transformed using:
  - Frequency encoding
  - Target encoding (with safeguards)
  - Hashing
  - Grouping rare categories into `"Other"`

---


### 1. Frequency Encoding

- Replace each category with its **occurrence count** or frequency.
- Keeps the feature numeric, avoiding one-hot explosion.

**Use case:**

Useful for tree-based models and linear models.


In [ ]:
import pandas as pd

# For simplisity, let's assume this is a large dataset
df = pd.DataFrame({
    'User': ['Alice', 'Bob', 'Charlie', 'David', 'Eve', 'Frank', 'Grace'],
    'City': ['New York', 'Los Angeles', 'New York', 'Chicago', 'Los Angeles', 'Chicago', 'Houston']
})
df

,User,City
0,User_697,City_45
1,User_529,City_31
2,User_153,City_22
3,User_926,City_31
4,User_150,City_5
...,...,...
1995,User_657,City_22
1996,User_186,City_31
1997,User_646,City_31
1998,User_97,City_9


In [2]:
freq = df['City'].value_counts()
df['City_freq'] = df['City'].map(freq)
df

,User,City,City_freq
0,User_697,City_45,8
1,User_529,City_31,372
2,User_153,City_22,722
3,User_926,City_31,372
4,User_150,City_5,227
...,...,...,...
1995,User_657,City_22,722
1996,User_186,City_31,372
1997,User_646,City_31,372
1998,User_97,City_9,9


### 2. Target Encoding (Mean Encoding)

- Replace each category with the mean target value (or other statistic) for that category.

- Adds predictive power for supervised tasks.

Important:

Must use cross-validation or out-of-fold encoding to avoid leakage.


In [ ]:
# For simplisity, let's assume this is a large dataset
df = pd.DataFrame({
    'User': ['Alice', 'Bob', 'Charlie', 'David', 'Eve', 'Frank', 'Grace'],
    'Occupation': ['Engineer', 'Doctor', 'Engineer', 'Artist', 'Doctor', 'Artist', 'Lawyer'],
    'Income': [70000, 120000, 75000, 40000, 115000, 45000, 90000]
})
df

,User,Occupation,Income
0,User_566,Occupation_46,89966.595619
1,User_867,Occupation_47,68991.524022
2,User_943,Occupation_21,68482.776877
3,User_674,Occupation_41,97884.787042
4,User_654,Occupation_27,102982.899987
...,...,...,...
1995,User_689,Occupation_47,68156.245741
1996,User_814,Occupation_13,80472.340304
1997,User_919,Occupation_6,91162.508678
1998,User_540,Occupation_25,83614.825835


In [4]:
df.Occupation.value_counts()

Occupation
Occupation_46    533
Occupation_47    356
Occupation_1     267
Occupation_44    126
Occupation_35     82
Occupation_25     54
Occupation_13     40
Occupation_45     36
Occupation_33     36
Occupation_20     25
Occupation_49     24
Occupation_42     22
Occupation_14     20
Occupation_9      17
Occupation_38     16
Occupation_28     16
Occupation_32     16
Occupation_18     15
Occupation_24     14
Occupation_48     14
Occupation_2      13
Occupation_36     13
Occupation_41     13
Occupation_37     12
Occupation_43     12
Occupation_5      12
Occupation_7      11
Occupation_31     11
Occupation_30     11
Occupation_29     11
Occupation_17     10
Occupation_10     10
Occupation_15      9
Occupation_34      9
Occupation_50      9
Occupation_6       8
Occupation_12      8
Occupation_16      8
Occupation_40      8
Occupation_23      8
Occupation_26      8
Occupation_19      8
Occupation_39      7
Occupation_27      7
Occupation_22      7
Occupation_21      7
Occupation_3       6
Oc

In [5]:
mean_income = df.groupby('Occupation')['Income'].mean()
df['Occupation_encoded'] = df['Occupation'].map(mean_income)
df


,User,Occupation,Income,Occupation_encoded
0,User_566,Occupation_46,89966.595619,87103.468883
1,User_867,Occupation_47,68991.524022,67800.169044
2,User_943,Occupation_21,68482.776877,70444.281144
3,User_674,Occupation_41,97884.787042,103299.272876
4,User_654,Occupation_27,102982.899987,104972.566031
...,...,...,...,...
1995,User_689,Occupation_47,68156.245741,67800.169044
1996,User_814,Occupation_13,80472.340304,80616.670597
1997,User_919,Occupation_6,91162.508678,89082.743839
1998,User_540,Occupation_25,83614.825835,88486.812300


### 3. Hashing

- Map categories into fixed number of buckets using a hash function.

- Reduces dimensionality and handles new/unseen categories gracefully.

Use case:

- Extremely high-cardinality categorical features

- Real-time systems


In [ ]:
from sklearn.feature_extraction import FeatureHasher

# For simplisity, let's assume this is a large dataset
df = pd.DataFrame({
    'Group_ID': ['U001', 'U002', 'U003', 'U004', 'U005', 'U006', 'U007'],
    'Activity': [5, 3, 8, 2, 7, 1, 4]
})
df

,Group_ID,Activity
0,U001,5
1,U002,3
2,U003,8
3,U004,2
4,U005,7
5,U006,1
6,U007,4


In [7]:
# Convert each User_ID to a list of strings (even if one element)
user_iterable = df['Group_ID'].apply(lambda x: [x])

In [8]:
hasher = FeatureHasher(n_features=5, input_type='string')

- `n_features=5` → output will have 5 hashed features (columns)

- Reduces dimensionality compared to one-hot encoding for many categories


In [9]:
hashed_features = hasher.transform(user_iterable)

hashed_df = pd.DataFrame(hashed_features.toarray(), columns=[f'hashed_{i}' for i in range(5)])
df = pd.concat([df, hashed_df], axis=1)
df

,Group_ID,Activity,hashed_0,hashed_1,hashed_2,hashed_3,hashed_4
0,U001,5,0.0,1.0,0.0,0.0,0.0
1,U002,3,0.0,0.0,0.0,0.0,1.0
2,U003,8,0.0,0.0,0.0,1.0,0.0
3,U004,2,0.0,0.0,0.0,0.0,1.0
4,U005,7,0.0,0.0,1.0,0.0,0.0
5,U006,1,0.0,0.0,0.0,0.0,-1.0
6,U007,4,0.0,0.0,0.0,0.0,1.0


#### Why not on-hot encoding?

Unlike `one-hot` encoding, where each unique category gets its own column, hashing maps all categories into a fixed number of columns (n_features). Each row (category) is represented as a sparse vector of mostly zeros, with a single `+1` or `-1` in the column determined by the hash function. The sign helps reduce the impact of collisions, which occur when multiple categories map to the same column. This approach allows high-cardinality features, like Group_ID or Product_ID, to be represented efficiently without creating thousands of one-hot columns, making it especially suitable for memory-efficient pipelines and real-time systems.


### 4. Grouping Rare Categories

- Combine infrequent categories into a single "Other" group.

- Reduces noise and prevents overfitting.

Use case:

- Marketing or product datasets with long-tail distributions


In [ ]:
# For simplisity, let's assume this is a large dataset

df = pd.DataFrame({
    'Product': ['A', 'B', 'C', 'A', 'D', 'E', 'B', 'F', 'G', 'A', 'H', 'C', 'I', 'J', 'K'],
    'Sales': [100, 50, 30, 120, 10, 5, 60, 8, 2, 110, 3, 35, 1, 2, 1]
})
df

,Product,Sales
0,A,100
1,B,50
2,C,30
3,A,120
4,D,10
5,E,5
6,B,60
7,F,8
8,G,2
9,A,110


In [11]:
# Count occurrences of each product
counts = df['Product'].value_counts()
counts


Product
A    3
B    2
C    2
D    1
E    1
F    1
G    1
H    1
I    1
J    1
K    1
Name: count, dtype: int64

In [12]:
# Group rare categories into "Other"
threshold = 2
df['Product_grouped'] = df['Product'].apply(lambda x: x if counts[x] >= threshold else 'Other')
df


,Product,Sales,Product_grouped
0,A,100,A
1,B,50,B
2,C,30,C
3,A,120,A
4,D,10,Other
5,E,5,Other
6,B,60,B
7,F,8,Other
8,G,2,Other
9,A,110,A
